# Reaction-Aware Bayesian Optimization for MOF Synthesis

This notebook runs the **reaction-aware** closed loop described in the manuscript on the Axo platform
(`science_jubilee.optimization.reaction_aware.ReactionAwareBOLoop`). It automates the workflow that was
run step by step in `sampling.ipynb`, `mof_synthesis.ipynb`, `gualtieri.ipynb` and `bayesian.ipynb`.

## Phases

| Phase | Name | Vials per batch | Measurement interval | Readings per vial |
|---|---|---|---|---|
| 0 | Initial sampling | 5 maximin seeds | 15 min | 13 (0–180 min) |
| 1 | Batch BO | 5 | 15 min | 13 (0–180 min) |
| 2 | Validation | 2 | 5 min | 13 (0–60 min) |

- **Objective:** nucleation rate constant `k_n = 1/a` from the Gualtieri fit at 400 nm, after the 400/590-nm optical
  gates (no reaction / no ZIF-67 feature score 0).
- **Phase rule:** after each batch, the extent of reaction at the first follow-up reading, α(t₁), is computed for every
  fitted vial. If any α(t₁) > 0.5 the run moves to phase 2 (one way).
- **Batch selection:** GP + Expected Improvement on the 8-mL / 0.1-mL lattice, xi = 0.01, exclusion radius 0.2,
  already-tested formulations excluded.

## Operator pauses

The loop saves its state and **stops** at every phase change and whenever no rack has enough unused vial
positions (two 10-vial racks, slots 2 and 5; a batch always stays in one slot). Re-run Part 5 and Part 6 to continue.

---
## Part 1: Setup and Imports

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add src to path
sys.path.insert(0, str(Path.cwd() / "src"))

from science_jubilee.Machine import Machine
from science_jubilee.Experiment import Experiment
from science_jubilee.tools.Syringe import Syringe
from science_jubilee.tools.Double_Syringe import DoubleSyringe
from science_jubilee.tools.Vacuum_Gripper import VacuumGripper
from science_jubilee.tools.Oceandirect_axo import Spectrometer

from science_jubilee.optimization.reaction_aware import ReactionAwareBOLoop, PHASES

print("✓ All imports successful")

---
## Part 2: Configuration

Choose your optimization mode and settings.

In [ ]:
# =============================================================================
# CONFIGURATION SETTINGS
# =============================================================================

# Machine IP address
JUBILEE_IP = "192.168.1.2"

# Operator information
OPERATOR_NAME = "YourName"  # Change this!

# Output directory for results (state, decisions.csv, vials.csv, plans/, plots/)
OUTPUT_DIR = "optimization_results"

# =============================================================================
# REACTION-AWARE SETTINGS (defaults follow the manuscript)
# =============================================================================

ALPHA_THRESHOLD = 0.5          # alpha(t1) above which the run moves to phase 2 (validation)
MAX_BO_BATCHES = 10            # stop if phase 1 runs this many batches without alpha(t1) > threshold
MAX_VALIDATION_BATCHES = 4     # number of 2-vial validation batches before the campaign ends
MAX_TOTAL_VIALS = None         # optional vial budget for the whole campaign
PAUSE_ON_PHASE_CHANGE = True   # stop at every phase change so the operator can check the run

# =============================================================================
# RESUME SETTINGS
# =============================================================================

RESUME_FROM_SAVED_STATE = False   # True to continue a paused or interrupted run
VIALS_REPLACED = None             # after a "vials full" pause: [] = all racks replaced, or e.g. [2]
DISCARD_PENDING = False           # True to drop a batch that was interrupted mid-measurement

# =============================================================================
# DISPLAY CONFIGURATION
# =============================================================================

print("Configuration:")
print(f"  Jubilee IP: {JUBILEE_IP}")
print(f"  Operator: {OPERATOR_NAME}")
print(f"  Output directory: {OUTPUT_DIR}")
for p, phase in PHASES.items():
    print(f"  Phase {p} ({phase.name}): {phase.batch_size} vials, {phase.interval_min}-min interval")
print(f"  alpha(t1) threshold: {ALPHA_THRESHOLD}")
print(f"  Max batch-BO batches: {MAX_BO_BATCHES}, validation batches: {MAX_VALIDATION_BATCHES}")
print(f"  Resume: {RESUME_FROM_SAVED_STATE}")

---
## Part 3: Hardware Initialization

### ⚠️ IMPORTANT: Manual Calibration Required

Before running this section, ensure you have:
1. **Removed labware from slots 0-1** (prevents collision during homing)
2. **Completed manual offset calibration** for all labware
3. **Saved offset files** in the labware definition directories

See `mof_synthesis.ipynb` for manual calibration procedure.

In [ ]:
# =============================================================================
# STEP 1: Initialize Machine
# =============================================================================

print("Connecting to Jubilee...")
machine = Machine(address=JUBILEE_IP)
print(f"✓ Connected to {JUBILEE_IP}")

print("\nHoming all axes...")
machine.home_all()
machine.move_to(z=200)  # Move to safe Z height
print("✓ Homing complete")

In [ ]:
# =============================================================================
# STEP 2: Load Deck
# =============================================================================

print("Loading deck configuration...")
deck = machine.load_deck("lab_automation_deck")
print("✓ Deck loaded")

In [ ]:
# =============================================================================
# STEP 3: Load Labware
# =============================================================================

print("Loading labware...\n")

# Slot 1: Precursors (metal + organic)
print("  Slot 1: Precursors")
precursors = machine.load_labware(
    'uwsunlab_2_wellplate_60000ul_slot1.json',
    slot=1,
    has_lid_on_top=False,
    currentLiquidVolume=60  # 60 mL initial volume
)
precursors.load_manualOffset()
print("    ✓ Precursors loaded with manual offsets")

# Slot 3: Solvents
print("  Slot 3: Solvents")
solvents = machine.load_labware(
    'uwsunlab_2_wellplate_60000ul_slot3.json',
    slot=3,
    has_lid_on_top=False,
    currentLiquidVolume=60
)
solvents.load_manualOffset()
print("    ✓ Solvents loaded with manual offsets")

# Slot 2: Sample vials (10-well plate, 3 views)
print("  Slot 2: Sample vials (3 views)")
samples2_ssy = machine.load_labware(
    'uwsunlab_10_wellplate_14000ul_ssy_1.json',
    slot=2,
    has_lid_on_top=False
)
samples2_ssy.load_manualOffset()

samples2_sy = machine.load_labware(
    'uwsunlab_10_wellplate_14000ul_sy_1.json',
    slot=2,
    has_lid_on_top=False
)
samples2_sy.load_manualOffset()

samples2_spec = machine.load_labware(
    'uwsunlab_10_wellplate_14000ul_spec_1.json',
    slot=2,
    has_lid_on_top=False
)
samples2_spec.load_manualOffset()
print("    ✓ Slot 2 samples loaded (ssy, sy, spec views)")

# Slot 5: Additional sample vials (10-well plate, 3 views)
print("  Slot 5: Additional sample vials (3 views)")
samples5_ssy = machine.load_labware(
    'uwsunlab_10_wellplate_14000ul_ssy_2.json',
    slot=5,
    has_lid_on_top=False
)
samples5_ssy.load_manualOffset()

samples5_sy = machine.load_labware(
    'uwsunlab_10_wellplate_14000ul_sy_2.json',
    slot=5,
    has_lid_on_top=False
)
samples5_sy.load_manualOffset()

samples5_spec = machine.load_labware(
    'uwsunlab_10_wellplate_14000ul_spec_2.json',
    slot=5,
    has_lid_on_top=False
)
samples5_spec.load_manualOffset()
print("    ✓ Slot 5 samples loaded (ssy, sy, spec views)")

print("\n✓ All labware loaded")

In [ ]:
# =============================================================================
# STEP 4: Load Tools
# =============================================================================

print("Loading tools...\n")

# Single syringe (T0)
print("  T0: Single Syringe")
single_syringe = Syringe(
    index=0,
    name='single_syringe',
    config='single_syringe'
)
machine.load_tool(single_syringe)
print("    ✓ Single syringe loaded")

# Dual syringe (T2)
print("  T2: Dual Syringe")
dual_syringe = DoubleSyringe(
    index=2,
    name='Dual_Syringe',
    config='10cc_syringe'
)
machine.load_tool(dual_syringe)
print("    ✓ Dual syringe loaded")

# Spectrometer (T3)
print("  T3: Spectrometer")
spectrometer = Spectrometer(
    index=3,
    name='Spectrometer',
    base_dir=r"C:\Axo\science-jubilee\axo\spectrum_data",
    experiment_name='BO_Optimization',  # Will be overwritten by orchestrator
    operator_name=OPERATOR_NAME,
    target_compound='ZIF',
    experiment_notes='Autonomous multi-phase Bayesian optimization',
    solvent='triethylamine/methanol',
    temperature_c=25
)
machine.load_tool(spectrometer)
print("    ✓ Spectrometer loaded")

# Vacuum gripper (T4)
print("  T4: Vacuum Gripper")
gripper = VacuumGripper(
    index=4,
    name='Vacuum_Gripper',
    vacuum_pin=0,
    limit_switch_pin=2
)
machine.load_tool(gripper)
print("    ✓ Vacuum gripper loaded")

print("\n✓ All tools loaded")

In [ ]:
# =============================================================================
# STEP 5: Configure Vacuum Gripper Locations
# =============================================================================

vacuum_location = {
    0: {"loc": (84, 51, 0), "labwares_list": []},
    1: {"loc": (225, 57, 0), "labwares_list": [precursors]},
    2: {"loc": (84, 147, 0), "labwares_list": [samples2_ssy, samples2_sy, samples2_spec]},
    3: {"loc": (225, 149, 0), "labwares_list": [solvents]},
    4: {"loc": (79, 244, 0), "labwares_list": []},
    5: {"loc": (221, 244, 0), "labwares_list": [samples5_ssy, samples5_sy, samples5_spec]}
}

print("✓ Vacuum gripper locations configured")

In [ ]:
# =============================================================================
# STEP 6: Organize Labware and Tools for Experiment
# =============================================================================

all_labwares = {
    "slot1": {"precursors": precursors},
    "slot2": {
        "samples2_ssy": samples2_ssy,
        "samples2_sy": samples2_sy,
        "samples2_spec": samples2_spec
    },
    "slot3": {"solvents": solvents},
    "slot4": {"vacuum_location": vacuum_location},
    "slot5": {
        "samples5_ssy": samples5_ssy,
        "samples5_sy": samples5_sy,
        "samples5_spec": samples5_spec
    }
}

all_tools = {
    "single_syringe": single_syringe,
    "dual_syringe": dual_syringe,
    "spectrometer": spectrometer,
    "gripper": gripper
}

print("✓ Labware and tools organized")

In [ ]:
# =============================================================================
# STEP 7: Create Experiment Instance
# =============================================================================

print("Creating Experiment instance...")
experiment = Experiment(
    machine=machine,
    deck=deck,
    all_tools=all_tools,
    all_labwares=all_labwares
)

print("✓ Experiment instance created")
print("\n" + "="*70)
print("HARDWARE SETUP COMPLETE")
print("="*70)

---
## Part 4: Initialize Orchestrator

Create the reaction-aware Bayesian optimization loop.

In [ ]:
# =============================================================================
# STEP 8: Phase overview
# =============================================================================

for p, phase in PHASES.items():
    print(f"Phase {p}: {phase.name}")
    print(f"  Vials per batch: {phase.batch_size}")
    print(f"  Measurement interval: {phase.interval_min} min x 13 readings")
print(f"\nPhase 1 -> 2 when any fitted vial has alpha(t1) > {ALPHA_THRESHOLD}")

In [ ]:
# =============================================================================
# STEP 9: Create Orchestrator
# =============================================================================

print("\nCreating reaction-aware Bayesian optimization loop...")

orchestrator = ReactionAwareBOLoop(
    experiment=experiment,
    output_dir=OUTPUT_DIR,
    operator_name=OPERATOR_NAME,
    alpha_threshold=ALPHA_THRESHOLD,
    max_bo_batches=MAX_BO_BATCHES,
    max_validation_batches=MAX_VALIDATION_BATCHES,
    max_total_vials=MAX_TOTAL_VIALS,
    pause_on_phase_change=PAUSE_ON_PHASE_CHANGE
)

if orchestrator.state_file.exists() and not RESUME_FROM_SAVED_STATE:
    raise RuntimeError(f"{orchestrator.state_file} exists: set RESUME_FROM_SAVED_STATE = True "
                       "or choose another OUTPUT_DIR.")

print("✓ Orchestrator created")
print(f"  Output directory: {orchestrator.output_dir}")
print(f"  State file: {orchestrator.state_file}")

---
## Part 5: Resume from Saved State (Optional)

**Skip this section if starting a new run.**

Run this cell after any pause:
- **Phase change:** just resume.
- **Vials full:** replace the used vials with empty ones, set `VIALS_REPLACED = []` (all racks) or e.g. `[2]`, then resume.
- **Interrupted batch:** inspect the vials, set `DISCARD_PENDING = True`, then resume.

In [ ]:
# =============================================================================
# OPTIONAL: Resume from Saved State
# =============================================================================

if RESUME_FROM_SAVED_STATE:
    print("Loading saved state...")
    orchestrator.load_state()
    if DISCARD_PENDING:
        orchestrator.discard_pending()
    if VIALS_REPLACED is not None:
        orchestrator.replace_vials(VIALS_REPLACED or None)

    s = orchestrator.summary()
    print("\nResumed state:")
    print(f"  Phase: {s['phase']} ({s['phase_name']})")
    print(f"  Batches so far: {s['batches']}, vials: {s['n_vials']}")
    print(f"  Unused vial positions: {s['unused_vial_positions']}")
    if s["best_k_n"] is not None:
        print(f"  Best k_n so far: {s['best_k_n']:.4f} min^-1 at Co/2-mIm/TEA = {s['best_composition_mL']} mL")
else:
    print("Starting new optimization run (no state loaded); racks in slots 2 and 5 must hold empty vials.")

---
## Part 6: Run Optimization Loop

### ⚠️ IMPORTANT: What to Expect

For every batch the loop will:

1. **Choose formulations**: the 5 maximin seeds first, then GP + EI (5 vials in phase 1, 2 in phase 2)
2. **Place the batch** on unused vial positions within one rack and write the synthesis plan
3. **Run the synthesis** and the timed UV–Vis readings (15-min or 5-min interval, 13 readings), writing a new data folder per batch
4. **Score every vial**: optical gates, Gualtieri fit at 400 nm, `k_n = 1/a`, and α(t₁)
5. **Apply the phase rule**: α(t₁) > 0.5 in any fitted vial moves the run to the validation phase
6. **Stop for the operator** at a phase change or when the racks are full (state is saved)

The first batch of every session asks the operator to switch the probe light off and on for the dark/white
reference spectra; later batches reuse them.

### Interrupting the Run

- Press **Kernel → Interrupt** to stop; the state file records the batch in progress
- To continue, set `RESUME_FROM_SAVED_STATE = True` (and `DISCARD_PENDING = True` if a batch was cut short) and re-run from Part 4

In [ ]:
# =============================================================================
# RUN THE OPTIMIZATION LOOP
# =============================================================================

print("=" * 70)
print("STARTING REACTION-AWARE BAYESIAN OPTIMIZATION")
print("=" * 70)

result = orchestrator.run()

print("\n" + "=" * 70)
print(result["status"].replace("_", " ").upper())
print("=" * 70)
print(result["message"])
print(f"Unused vial positions: {result['unused_vial_positions']}")

---
## Part 7: Results Analysis

Analyze the optimization results.

In [ ]:
# =============================================================================
# VIEW RESULTS
# =============================================================================

if len(orchestrator.y):
    best_idx = int(np.argmax(orchestrator.y))
    best_comp = orchestrator.X[best_idx]
    print("RESULTS SO FAR")
    print("=" * 70)
    print(f"\nBest observed k_n: {orchestrator.y[best_idx]:.4f} min^-1")
    print(f"  Co²⁺ volume: {best_comp[0]:.1f} mL")
    print(f"  2-methylimidazole volume: {best_comp[1]:.1f} mL")
    print(f"  Triethylamine volume: {best_comp[2]:.1f} mL")
    print(f"\nVials: {len(orchestrator.y)}, batches: {orchestrator.iteration}, "
          f"current phase: {orchestrator.phase} ({PHASES[orchestrator.phase].name}), status: {orchestrator.status}")

    decisions = pd.read_csv(orchestrator.output_dir / "decisions.csv")
    display(decisions[["iteration", "experiment_name", "phase", "n_vials", "interval_min",
                       "max_alpha_t1", "next_phase", "best_k_n_so_far"]])
else:
    print("No results available yet")

In [ ]:
# =============================================================================
# PLOT OPTIMIZATION PROGRESS
# =============================================================================

vials_csv = orchestrator.output_dir / "vials.csv"
if vials_csv.exists():
    vials = pd.read_csv(vials_csv)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # k_n per vial and best so far, coloured by phase
    ax = axes[0]
    colors = {0: "tab:blue", 1: "tab:orange", 2: "tab:red"}
    order = np.arange(len(vials))
    for p, group in vials.groupby("phase"):
        ax.scatter(order[group.index], group["optimization_score"], color=colors[p],
                   label=f"phase {p} ({PHASES[p].name})")
    ax.plot(order, np.maximum.accumulate(vials["optimization_score"]), "k-", label="best so far")
    ax.set_xlabel("Vial number")
    ax.set_ylabel("k_n (min$^{-1}$)")
    ax.set_title("Optimization progress")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # alpha(t1) per batch against the threshold
    ax = axes[1]
    decisions = pd.read_csv(orchestrator.output_dir / "decisions.csv")
    ax.bar(decisions["iteration"], decisions["max_alpha_t1"].fillna(0),
           color=[colors[p] for p in decisions["phase"]])
    ax.axhline(ALPHA_THRESHOLD, color="k", linestyle="--", label=f"threshold {ALPHA_THRESHOLD}")
    ax.set_xlabel("Batch")
    ax.set_ylabel("max alpha(t$_1$)")
    ax.set_title("Phase rule")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    output_fig = orchestrator.output_dir / "optimization_summary.png"
    plt.savefig(output_fig, dpi=300, bbox_inches="tight")
    print(f"Figure saved to: {output_fig}")
    plt.show()
else:
    print("No data available for plotting")

In [ ]:
# =============================================================================
# VIEW PER-VIAL RESULTS
# =============================================================================

if vials_csv.exists():
    display(vials[["iteration", "phase", "batch", "slot", "vial_id", "Co_mL", "mIm_mL", "TEA_mL",
                   "status", "k_n_min_inverse", "alpha_first_followup", "optimization_score"]])
else:
    print("No per-vial results yet")

---
## Part 8: Cleanup and Shutdown

Safely shutdown the system.

In [ ]:
# =============================================================================
# CLEANUP
# =============================================================================

print("Cleaning up...")

# Park current tool
try:
    machine.park_tool()
    print("✓ Tool parked")
except:
    print("  (No tool was active)")

# Move to safe position
machine.move_to(z=200)
machine.move_to(x=0, y=0)
print("✓ Moved to home position")

# Save final state
orchestrator.save_state()
print(f"✓ Final state saved to {orchestrator.state_file}")

print("\n" + "="*70)
print("CLEANUP COMPLETE")
print("="*70)
print(f"\nAll results saved to: {orchestrator.output_dir}")
print("Check the following files:")
print(f"  - {orchestrator.state_file} (resume state)")
print(f"  - {orchestrator.output_dir}/bo_optimization.log (detailed log)")
print(f"  - {orchestrator.output_dir}/decisions.csv (phase decisions per batch)")
print(f"  - {orchestrator.output_dir}/vials.csv (k_n and alpha(t1) per vial)")
print(f"  - {orchestrator.output_dir}/plans/ (synthesis plans)")

---
## Troubleshooting

### Common Issues

1. **"Error: machine must first be homed"**
   - Solution: Re-run Part 3, Step 1 (machine.home_all())

2. **"Well cannot accommodate X ml dispense liquid volume"**
   - Solution: Check currentLiquidVolume initialization in Part 3, Step 3

3. **"Lid is on top of wellPlate"**
   - Solution: Set `labware.has_lid_on_top = False` manually

4. **Status `paused_vials_full`**
   - Replace the used vials with empty ones, set `VIALS_REPLACED = []` (or the slot list) and resume

5. **Status `paused_pending_incomplete`**
   - A batch stopped before all readings were taken. Inspect it, set `DISCARD_PENDING = True` and resume
   - Its vial positions stay marked as used until vials are replaced

6. **Status `paused_analysis_failed`**
   - The batch data could not be analysed (missing log events, recipe mismatch). Fix the data and resume,
     or discard the batch with `DISCARD_PENDING = True`

### Getting Help

- Check detailed logs: `{OUTPUT_DIR}/bo_optimization.log`
- Every decision is in `{OUTPUT_DIR}/decisions.csv`; every vial in `{OUTPUT_DIR}/vials.csv`
- The legacy I_max three-phase orchestrator is still available: `run_bo_optimization.py --policy phases`